# 05 - Statistical Analysis, Hypothesis Testing & Thesis Publication Figures

This notebook brings together the **Statistical Hypothesis Testing Engine** and the **Publication Figure & Storyboard Generator** into a unified scientific analysis pipeline.

### 🔬 Part I: Statistical Hypothesis Testing & Reporting
1. **Omnibus Kruskal-Wallis Test**: Multi-group non-parametric difference test across all solvers per problem condition.
2. **Pairwise Mann-Whitney U Tests with FDR**: Two-sided comparisons with **Benjamini-Hochberg False Discovery Rate** correction ($\alpha = 0.05$).
3. **Effect Size Estimation**: Vargha-Delaney $\hat{A}_{12}$ stochastic dominance metric.
4. **Synthesis Transfer Correlation**: Pearson $r, p$ connecting synthesis fitness to empirical benchmark error.
5. **Master Markdown Report**: Automated scientific summary exported to `results/reports/comprehensive_master_report.md`.

---
### 📊 Part II: Thesis Publication Figures & Visual Storyboard
- **Figure E**: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension).
- **Figure 1 (RQ1)**: Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2/RQ3)**: Empirical Convergence Trajectories & Target Precision ECDFs with IQR shaded bounds.
- **Figure 3 (RQ3 Hero)**: Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation)**: Prompt Scaffolding Ablation across LLM model families.

In [1]:
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Setup paths, libraries, and services
%load_ext autoreload
%autoreload 2

import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.config import DATA_DIR, RESULTS_DIR
from shared.database import create_db_session_factory
from benchmarking.infra.storage import SQLiteBenchmarkReadRepository
from benchmarking.application.statistical_service import StatisticalEvaluationService
from benchmarking.infra.io.trace_repository import IOHTraceReader
from benchmarking.domain.taxonomy import (
    BBOB_CLASSES,
    BBOB_NAMES,
    get_bbob_class,
    get_bbob_name,
)
from benchmarking.domain.resolvers import (
    resolve_canonical_model_slug,
    resolve_folder_solver_name,
)

session_factory = create_db_session_factory()
sqlite_repo = SQLiteBenchmarkReadRepository(session_factory)
service = StatisticalEvaluationService(sqlite_repo=sqlite_repo, trace_repo=trace_reader)
trace_reader = IOHTraceReader()

EVALUATIONS_DIR   = RESULTS_DIR / 'evaluations' / 'traces'
PUBLICATION_DIR   = RESULTS_DIR / 'publication'
EVAL_PROFILES_DIR = RESULTS_DIR / 'evaluations' / 'visual_profiles'
STATISTICS_DIR    = RESULTS_DIR / 'statistics'
REPORTS_DIR       = RESULTS_DIR / 'reports'

PUBLICATION_DIR.mkdir(parents=True, exist_ok=True)
EVAL_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
STATISTICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def comparative_dir(dim: int) -> Path:
    p = PUBLICATION_DIR / f'{dim}D'
    p.mkdir(parents=True, exist_ok=True)
    return p

def model_fig_dir(folder_name: str, dim: int) -> Path:
    slug = resolve_canonical_model_slug(folder_name)
    p = EVAL_PROFILES_DIR / slug / f'{dim}D'
    p.mkdir(parents=True, exist_ok=True)
    return p

def model_std_dir(model_slug: str, dim: int, noise_std: float) -> Path:
    slug = resolve_canonical_model_slug(model_slug)
    p = EVAL_PROFILES_DIR / slug / f'{dim}D' / f'std_{noise_std}'
    p.mkdir(parents=True, exist_ok=True)
    return p

def build_dynamic_solver_palette(solvers: list[str]) -> dict[str, str]:
    """Algorithmically assign publication-grade colors to dynamically discovered solvers."""
    STRATEGY_COLORS = {
        'baseline':      '#1f77b4',
        'guided':        '#ff7f0e',
        'thinking':      '#2ca02c',
        'vectorization': '#d62728',
    }
    CLASSICAL_COLORS = {
        'CMA-ES': '#8c564b',
        'DE':     '#e377c2',
        'PSO':    '#7f7f7f',
    }
    QUALITATIVE_CYCLE = pc.qualitative.Plotly + pc.qualitative.Dark24 + pc.qualitative.Set1
    palette = {}
    fallback_idx = 0
    for s in solvers:
        if s in CLASSICAL_COLORS:
            palette[s] = CLASSICAL_COLORS[s]
        elif ' / ' in s:
            model_tag, strat = s.split(' / ', 1)
            strat_lower = strat.lower()
            if '14B' in model_tag and strat_lower in STRATEGY_COLORS:
                palette[s] = STRATEGY_COLORS[strat_lower]
            elif '7B' in model_tag and strat_lower == 'baseline':
                palette[s] = '#9467bd'
            elif strat_lower in STRATEGY_COLORS and '14B' not in model_tag:
                palette[s] = QUALITATIVE_CYCLE[fallback_idx % len(QUALITATIVE_CYCLE)]
                fallback_idx += 1
            else:
                palette[s] = QUALITATIVE_CYCLE[fallback_idx % len(QUALITATIVE_CYCLE)]
                fallback_idx += 1
        else:
            palette[s] = QUALITATIVE_CYCLE[fallback_idx % len(QUALITATIVE_CYCLE)]
            fallback_idx += 1
    return palette

# ── Optional User Filters ──────────────────────────────────────────────────
FILTER_MODELS     = None   # e.g., ['14b'], ['7b'], or None for all
FILTER_PROBLEMS   = None   # e.g., [1, 8], [8, 11, 15], or None for all
FILTER_STRATEGIES = None   # e.g., ['baseline', 'guided'], or None for all
FILTER_DIMS       = None   # e.g., [2, 3], [5], or None for all
FILTER_NOISE_STDS = None   # e.g., [0.0, 0.05], or None for all

print('✅ Statistical analysis & figure generation environment initialized.')

✅ Statistical analysis & figure generation environment initialized.


## Part I: Statistical Hypothesis Testing & Reporting
Ingest empirical benchmark traces and conduct rigorous non-parametric hypothesis testing with FDR control.

In [2]:
# Ingest benchmark traces and synthesis records
df_exp, df_iter = service.get_synthesis_dataframes()
all_benchmark_data = service.load_evaluation_traces(
    dims=FILTER_DIMS,
    problems=FILTER_PROBLEMS,
    noise_stds=FILTER_NOISE_STDS,
    solver_resolver=resolve_folder_solver_name,
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark traces found in {EVALUATIONS_DIR}!')

all_dims = sorted(list(set(k[0] for k in all_benchmark_data.keys())))
all_noise_stds = sorted(list(set(k[1] for k in all_benchmark_data.keys())))
clean_std = 0.0 if 0.0 in all_noise_stds else (all_noise_stds[0] if all_noise_stds else 0.0)
noisy_std = next((n for n in all_noise_stds if n > 0.0), all_noise_stds[-1] if all_noise_stds else 0.05)
PROBLEM_IDS = sorted(list(set(k[2] for k in all_benchmark_data.keys())))

DISCOVERED_SOLVERS = sorted(list(set(s for cond in all_benchmark_data.values() for s in cond.keys())))
SOLVER_PALETTE = build_dynamic_solver_palette(DISCOVERED_SOLVERS)

MODELS_TO_SOLVERS = defaultdict(list)
for s in DISCOVERED_SOLVERS:
    if ' / ' in s:
        MODELS_TO_SOLVERS[s.split(' / ')[0]].append(s)

LLM_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' in s]
CLASSICAL_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' not in s]
ALL_SOLVERS_ORDER = LLM_SOLVERS_ORDER + CLASSICAL_SOLVERS_ORDER

print(f'📦 Loaded {len(df_exp)} experiments and {len(all_benchmark_data)} problem conditions.')
print(f'🎯 Problems: {PROBLEM_IDS} | Dimensions: {all_dims} | Solvers: {DISCOVERED_SOLVERS}')


📦 Loaded 292 experiments and 30 problem conditions.
🎯 Problems: [1, 8, 11, 15, 21] | Dimensions: [2, 3, 5] | Solvers: ['CMA-ES', 'DE', 'LLaMEA-14B / baseline', 'LLaMEA-14B / guided', 'LLaMEA-14B / thinking', 'LLaMEA-14B / vectorization', 'LLaMEA-7B / baseline', 'LLaMEA-7B / guided', 'LLaMEA-7B / thinking', 'LLaMEA-7B / vectorization', 'PSO']


In [3]:
# ── 1. Omnibus Kruskal-Wallis & Pairwise FDR Tests ─────────────────────────
df_omnibus = service.run_omnibus_kruskal(all_benchmark_data)
df_pairwise = service.run_pairwise_fdr(all_benchmark_data, alpha=0.05)
r_val, p_val = service.compute_synthesis_transfer_correlation(df_exp)

print(f'✅ Omnibus Tests: {len(df_omnibus)} rows ({len(df_omnibus[df_omnibus["Significant"] == "Yes"])} significant)')
print(f'✅ Pairwise FDR Tests: {len(df_pairwise)} rows ({len(df_pairwise[df_pairwise["Significant (FDR)"]])} significant)')
print(f'✅ Synthesis Transfer Correlation: r = {r_val:.3f} (p = {p_val:.3e})')

# Export Master Markdown Report
report_path = REPORTS_DIR / 'comprehensive_master_report.md'
service.generate_markdown_report(df_omnibus=df_omnibus, df_pairwise=df_pairwise, df_exp=df_exp, output_path=report_path)
print(f'🎉 Master Comprehensive Report generated: {report_path}')


✅ Omnibus Tests: 30 rows (30 significant)
✅ Pairwise FDR Tests: 1630 rows (1241 significant)
✅ Synthesis Transfer Correlation: r = 0.000 (p = 1.000e+00)
🎉 Master Comprehensive Report generated: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports/comprehensive_master_report.md


## Part II: Thesis Publication Figures & Visual Storyboard
Render high-DPI thesis figures and storyboard artifacts.

# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
clean_std = 0.0
noisy_std = 0.05

for dim in all_dims:


In [4]:
# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
for dim in all_dims:
    fig_e = make_subplots(
        rows=2, cols=2,
        specs=[[{}, {}], [{'colspan': 2}, None]],
        subplot_titles=(
            f'<b>(A1) Mean Success Rate — Clean (σ={clean_std}, {dim}D)</b>',
            f'<b>(A2) Mean Success Rate — Noisy (σ={noisy_std}, {dim}D)</b>',
            f'<b>(B) Landscape Fragility Index Matrix (Clean → Noisy σ={noisy_std} Degradation, {dim}D)</b>'
        ),
        vertical_spacing=0.18,
        horizontal_spacing=0.08,
        row_heights=[0.45, 0.55]
    )

    # (A1) & (A2): Data from service
    for c_idx, (noise_lvl, sub_col) in enumerate([(clean_std, 1), (noisy_std, 2)], start=1):
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, ALL_SOLVERS_ORDER, noise_level=noise_lvl)
        for solver in ALL_SOLVERS_ORDER:
            sub_s = df_hard[df_hard['Solver'] == solver]
            if not sub_s.empty:
                fig_e.add_trace(
                    go.Bar(
                        name=solver,
                        x=sub_s['Class'],
                        y=sub_s['Success Rate'],
                        marker_color=SOLVER_PALETTE.get(solver, '#666666'),
                        showlegend=(c_idx == 1)
                    ),
                    row=1, col=sub_col
                )

    # (B): Fragility Matrix from service
    frag_matrix, p_labels = service.compute_fragility_matrix(
        all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig_e.add_trace(
        go.Heatmap(
            z=frag_matrix,
            x=ALL_SOLVERS_ORDER,
            y=p_labels,
            colorscale='RdBu',
            zmid=0,
            colorbar=dict(
                title='<b>Fragility Δ</b>',
                thickness=12,
                len=0.45,
                y=0.22,
                yanchor='middle'
            )
        ),
        row=2, col=1
    )

    fig_e.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Figure E: Problem Difficulty & Noise Sensitivity Dashboard — {dim}D</b><br><sup>Cross-Environment Robustness Breakdown and Fragility Index Matrix by BBOB Landscape Class</sup>',
            x=0.02, y=0.98,
            font=dict(size=14, color='#2c3e50')
        ),
        barmode='group',
        width=1150, height=800,
        margin=dict(l=70, r=40, t=100, b=70),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
        )
    )

    out_path = comparative_dir(dim) / 'figure_e_difficulty_and_noise.png'
    fig_e.write_image(str(out_path), scale=3)

print('✅ Figure E generated for all dimensions.')


2026-08-24 23:31:24 INFO Chromium init'ed with kwargs {}
2026-08-24 23:31:24 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 23:31:24 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmprne8u1oo.
2026-08-24 23:31:24 INFO Opening browser.
2026-08-24 23:31:24 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmphkeoq188.
2026-08-24 23:31:24 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmphkeoq188
2026-08-24 23:31:26 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmprne8u1oo/index.html
2026-08-24 23:31:27 INFO Getting tab from queue (has 1)
2026-08-24 23:31:27 INFO Got 49EF
2026-08-24 23:31:27 INFO Reloading tab 49EF before return.
2026-08-24 23:31:28 INFO Putting tab 49EF back (queue size: 0).
2026-08-24 23:31:28 INFO Waiting for all cleanups to finish.
2026-08-24 23:31:28 INFO Exiting Kaleido.
2026-08-24 23:31:28 INFO T

✅ Figure E generated for all dimensions.


### 📊 Model-Specific Hardness Success Rates (Clean vs. Noisy)
Separates the mean success rate analysis per LLM model (, ) across clean and noisy landscapes.

In [5]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'<b>(A) Clean Landscape (σ={clean_std}, {dim}D)</b>',
            f'<b>(B) Noisy Landscape (σ={noisy_std}, {dim}D)</b>'
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        for solver in solvers_list:
            sub_s = df_hard[df_hard['Solver'] == solver]
            if not sub_s.empty:
                fig.add_trace(
                    go.Bar(
                        name=solver,
                        x=sub_s['Class'],
                        y=sub_s['Success Rate'],
                        marker_color=SOLVER_PALETTE.get(solver, '#666666'),
                        showlegend=(c_idx == 1)
                    ),
                    row=1, col=c_idx
                )
                
    fig.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Mean Success Rate by Landscape Hardness — {model_tag} ({dim}D)</b><br><sup>Empirical Success Rate Across Problem Classes</sup>',
            x=0.02, y=0.96,
            font=dict(size=14, color='#2c3e50')
        ),
        barmode='group',
        width=1050, height=460,
        margin=dict(l=60, r=30, t=90, b=60),
        yaxis=dict(range=[0, 1.05], title='<b>Success Rate</b>'),
        yaxis2=dict(range=[0, 1.05]),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
        )
    )
    
    out_p = model_fig_dir(model_tag, dim) / 'figure_success_rate_by_hardness.png'
    fig.write_image(str(out_p), scale=3)

for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        solvers_to_plot = solvers_list + CLASSICAL_SOLVERS_ORDER
        render_model_success_rate_by_hardness(model_name, solvers_to_plot, dim)

print('✅ Model-specific success rate by hardness generated for all models and dimensions.')


2026-08-24 23:31:32 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:31:32 INFO shutil.rmtree worked.
2026-08-24 23:31:32 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:31:32 INFO shutil.rmtree worked.
2026-08-24 23:31:32 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:31:32 INFO shutil.rmtree worked.
2026-08-24 23:31:32 INFO Chromium init'ed with kwargs {}
2026-08-24 23:31:32 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 23:31:32 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5x4uygm0.
2026-08-24 23:31:32 INFO Opening browser.
2026-08-24 23:31:32 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp1b1oyj3q.
2026-08-24 23:31:32 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp1b1oyj3q
2026-08-24 23:31:33 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5x4uygm0/index.html
2026-08-24 23:3

✅ Model-specific success rate by hardness generated for all models and dimensions.


# 🎓 Part II: Thesis Visual Storyboard (RQ1 → RQ2 → RQ3 → Scaffolding Narrative Chain)

The following four figures form the core visual evidence for the thesis, saved into `results/figures/{dim}D/thesis/`:
- **Figure 1 (RQ1):** Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2):** LLaMEA Synthesis Competency vs. Classical Baselines (Clean Convergence Trajectories & IQR).
- **Figure 3 (RQ3 Hero):** Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation):** Prompt Scaffolding Ablation on LLaMEA-14B (Baseline vs. Guided vs. Thinking vs. Vectorization).


In [6]:
# ── THESIS Figure 1: Benchmark Validation (RQ1: Stochastic Extension) ─────────
for dim in all_dims:
    clean_medians, noisy_medians, problem_labels = service.compute_validation_medians(
        all_benchmark_data, dim, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        name=f'<b>Clean Evaluation (σ={clean_std})</b>',
        x=problem_labels,
        y=np.maximum(clean_medians, 1e-16),
        marker=dict(color='#2B5C8F', line=dict(color='#1B3A5B', width=1.5))
    ))
    fig1.add_trace(go.Bar(
        name=f'<b>Noisy Evaluation (σ={noisy_std})</b>',
        x=problem_labels,
        y=np.maximum(noisy_medians, 1e-16),
        marker=dict(
            color='#D95F02',
            pattern=dict(shape='/', fillmode='replace', fgcolor='#FFFFFF', fgopacity=0.35, size=8),
            line=dict(color='#8C3800', width=1.5)
        )
    ))

    fig1.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Figure 1: Benchmark Problem Difficulty Under Stochastic Noise Extension — {dim}D</b><br><sup>Median Terminal Optimization Precision (Δy) Across Solvers by Landscape Class</sup>',
            x=0.02, y=0.96,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        xaxis=dict(title='<b>BBOB Landscape Class</b>', tickfont=dict(size=11)),
        yaxis=dict(
            type='log', title='<b>Median Final Error log₁₀(Δy)</b>', range=[-16, 4],
            showgrid=True, gridwidth=1, gridcolor='#EAEAEA'
        ),
        barmode='group', bargap=0.25, bargroupgap=0.1,
        width=950, height=540,
        margin=dict(l=65, r=30, t=95, b=65),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=12, color='#333333'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
        )
    )

    out_p = comparative_dir(dim) / 'figure_1_benchmark_validation.png'
    fig1.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 1 generated for all dimensions in results/publication/{dim}D/.')


2026-08-24 23:31:45 INFO Chromium init'ed with kwargs {}
2026-08-24 23:31:45 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 23:31:45 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6t44rfxo.
2026-08-24 23:31:45 INFO Opening browser.
2026-08-24 23:31:45 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpymbhy7w8.
2026-08-24 23:31:45 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpymbhy7w8
2026-08-24 23:31:46 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6t44rfxo/index.html
2026-08-24 23:31:47 INFO Getting tab from queue (has 1)
2026-08-24 23:31:47 INFO Got 98D2
2026-08-24 23:31:47 INFO Reloading tab 98D2 before return.
2026-08-24 23:31:47 INFO Putting tab 98D2 back (queue size: 0).
2026-08-24 23:31:47 INFO Waiting for all cleanups to finish.
2026-08-24 23:31:47 INFO Exiting Kaleido.
2026-08-24 23:31:47 INFO T

✅ Thesis Figure 1 generated for all dimensions in results/publication/{dim}D/.


In [7]:
# ── THESIS Figure 2: Empirical Convergence Trajectories & ECDFs (RQ2 & RQ3) ──
eval_grid = np.logspace(0, 5, 200)
targets = np.logspace(-8, 2, 100)

def get_dynamic_grid(n_problems: int, max_cols: int = 3):
    n_total = n_problems + 1  # problems + 1 overall ECDF
    n_cols = min(max_cols, n_total)
    n_rows = int(np.ceil(n_total / n_cols))
    coords = [((i // n_cols) + 1, (i % n_cols) + 1) for i in range(n_total)]
    return coords, n_rows, n_cols

def render_figure_2_trajectories(model_slug: str, solvers_to_plot: list, dim: int, noise_std: float, label_env: str):
    coords, n_rows, n_cols = get_dynamic_grid(len(PROBLEM_IDS), max_cols=3)
    subplot_titles = [
        f"<b>{BBOB_NAMES.get(p, f'f{p}')}</b><br><sup>{BBOB_CLASSES.get(p, '')}</sup>"
        for p in PROBLEM_IDS
    ] + ["<b>Overall ECDF (Target Precision)</b><br><sup>Empirical CDF</sup>"]

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.10, horizontal_spacing=0.06
    )

    overall_ecdfs = {s: [] for s in solvers_to_plot}

    for idx, p_id in enumerate(PROBLEM_IDS):
        row, col = coords[idx]
        key = (dim, noise_std, p_id)
        solvers_dict = all_benchmark_data.get(key, {})

        for s_name in solvers_to_plot:
            runs = solvers_dict.get(s_name, [])
            med, q25, q75, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
            overall_ecdfs[s_name].append(ecdf_curve)

            if np.isnan(med).all():
                continue
            color = SOLVER_PALETTE.get(s_name, "#666666")

            fig.add_trace(
                go.Scatter(
                    x=eval_grid, y=med, mode="lines", name=s_name,
                    line=dict(color=color, width=2.0),
                    showlegend=(idx == 0)
                ),
                row=row, col=col
            )
            fig.add_trace(
                go.Scatter(
                    x=np.concatenate([eval_grid, eval_grid[::-1]]),
                    y=np.concatenate([q75, q25[::-1]]),
                    fill="toself",
                    fillcolor=color.replace("rgb", "rgba").replace(")", ", 0.12)"),
                    line=dict(color="rgba(255,255,255,0)"),
                    showlegend=False, hoverinfo="skip"
                ),
                row=row, col=col
            )

        fig.update_xaxes(type="log", title_text="<b>Evaluations</b>", showgrid=True, gridcolor="#F0F0F0", row=row, col=col)
        fig.update_yaxes(type="log", title_text="<b>Δy</b>", range=[-16, 4], showgrid=True, gridcolor="#F0F0F0", row=row, col=col)

    # Last subplot: Overall ECDF
    ecdf_row, ecdf_col = coords[-1]
    for s_name in solvers_to_plot:
        curves = overall_ecdfs[s_name]
        if curves:
            mean_ecdf = np.mean(curves, axis=0)
            color = SOLVER_PALETTE.get(s_name, "#666666")
            fig.add_trace(
                go.Scatter(
                    x=targets, y=mean_ecdf, mode="lines", name=s_name,
                    line=dict(color=color, width=2.2),
                    showlegend=False
                ),
                row=ecdf_row, col=ecdf_col
            )

    fig.update_xaxes(type="log", title_text="<b>Target Precision Δy</b>", autorange="reversed", showgrid=True, gridcolor="#F0F0F0", row=ecdf_row, col=ecdf_col)
    fig.update_yaxes(title_text="<b>Proportion of Solved Runs</b>", range=[-0.02, 1.05], showgrid=True, gridcolor="#F0F0F0", row=ecdf_row, col=ecdf_col)

    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 2: Empirical Optimization Trajectories ({label_env}) — {model_slug.upper()} ({dim}D)</b><br><sup>Log-scale Convergence Trajectories with Shaded IQR (25th–75th Percentiles) and Empirical Target Precision ECDF</sup>",
            x=0.02, y=0.98,
            font=dict(size=14, color="#2c3e50")
        ),
        width=1240, height=380 * n_rows,
        margin=dict(l=60, r=40, t=100, b=60),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0,
            bgcolor="rgba(255,255,255,0.9)", bordercolor="rgba(0,0,0,0.15)", borderwidth=1
        )
    )

    out_dir = model_std_dir(model_slug, dim, noise_std)
    out_trajectories = out_dir / "convergence_trajectories.png"
    fig.write_image(str(out_trajectories), scale=3)

for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        solvers_to_plot = solvers_list + CLASSICAL_SOLVERS_ORDER
        for n_std, label_env in [(clean_std, "Clean"), (noisy_std, "Noisy")]:
            render_figure_2_trajectories(model_name, solvers_to_plot, dim, n_std, label_env)

print("✅ Thesis Figure 2 generated for all models, regimes, and dimensions.")


2026-08-24 23:31:51 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:31:51 INFO shutil.rmtree worked.
2026-08-24 23:31:51 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:31:51 INFO shutil.rmtree worked.
2026-08-24 23:31:51 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:31:51 INFO shutil.rmtree worked.
2026-08-24 23:31:51 INFO Chromium init'ed with kwargs {}
2026-08-24 23:31:51 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 23:31:51 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp2oua2du2.
2026-08-24 23:31:51 INFO Opening browser.
2026-08-24 23:31:51 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5oc9kydx.
2026-08-24 23:31:51 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5oc9kydx
2026-08-24 23:31:52 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp2oua2du2/index.html
2026-08-24 23:3

✅ Thesis Figure 2 generated for all models, regimes, and dimensions.


In [8]:
# ── THESIS Figure 3: Cross-Environment Noise Robustness Profile (RQ3 Hero) ────
for dim in all_dims:
    fig_rob = go.Figure()
    valid_solvers, clean_rates, noisy_rates, deltas = service.compute_robustness_profile(
        all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )

    fig_rob.add_trace(go.Bar(
        name=f'<b>Clean Environment (σ={clean_std})</b>',
        x=valid_solvers,
        y=clean_rates,
        marker=dict(color='#2B5C8F', line=dict(color='#1B3A5B', width=1.5))
    ))
    fig_rob.add_trace(go.Bar(
        name=f'<b>Noisy Environment (σ={noisy_std})</b>',
        x=valid_solvers,
        y=noisy_rates,
        marker=dict(
            color='#D95F02',
            pattern=dict(shape='/', fillmode='replace', fgcolor='#FFFFFF', fgopacity=0.35, size=8),
            line=dict(color='#8C3800', width=1.5)
        )
    ))

    for s, c_r, n_r, delta in zip(valid_solvers, clean_rates, noisy_rates, deltas):
        drop_pct = (delta / c_r * 100) if c_r > 0 else 0.0
        fig_rob.add_annotation(
            x=s, y=max(c_r, n_r) + 0.04,
            text=f'<b>-Δ{drop_pct:.0f}%</b>' if delta > 0 else '<b>0%</b>',
            showarrow=False,
            font=dict(size=11, color='#D95F02' if drop_pct > 25 else '#4A7C59')
        )

    fig_rob.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Figure 3: Cross-Environment Noise Robustness Profile — {dim}D</b><br><sup>Generalization Retention: Clean (σ={clean_std}) vs. Stochastic (σ={noisy_std}) Overall Target Success Rate (Δy ≤ 10⁻⁸)</sup>',
            x=0.02, y=0.96,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        xaxis=dict(title='<b>Optimization Solver</b>', tickangle=-25, tickfont=dict(size=11)),
        yaxis=dict(
            title='<b>Overall Target Success Rate</b>', range=[0, 1.15],
            showgrid=True, gridwidth=1, gridcolor='#EAEAEA'
        ),
        barmode='group', bargap=0.25, bargroupgap=0.1,
        width=1000, height=560,
        margin=dict(l=65, r=30, t=95, b=90),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
        )
    )

    out_p = comparative_dir(dim) / 'figure_3_robustness.png'
    fig_rob.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 3 generated for all dimensions.')


2026-08-24 23:32:20 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 23:32:20 INFO shutil.rmtree worked.
2026-08-24 23:32:20 INFO Chromium init'ed with kwargs {}
2026-08-24 23:32:20 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 23:32:20 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpr15nk3k4.
2026-08-24 23:32:20 INFO Opening browser.
2026-08-24 23:32:20 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpnwy7x5si.
2026-08-24 23:32:20 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpnwy7x5si
2026-08-24 23:32:21 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpr15nk3k4/index.html
2026-08-24 23:32:22 INFO Getting tab from queue (has 1)
2026-08-24 23:32:22 INFO Got BED6
2026-08-24 23:32:22 INFO Reloading tab BED6 before return.
2026-08-24 23:32:22 INFO Putting tab BED6 back (queue size: 0).
2026-08-24 23:32:22 

✅ Thesis Figure 3 generated for all dimensions.


In [9]:
# ── THESIS Figure 4: Prompt Scaffolding Ablation Across Discovered Models (RQ2/3) ──
for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        if len(solvers_list) <= 1:
            continue  # Skip single-strategy models

        strat_labels, clean_succ_rates, noisy_succ_rates = service.compute_scaffolding_ablation(
            all_benchmark_data, dim, solvers_list, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
        )

        fig4 = go.Figure()
        fig4.add_trace(go.Bar(
            name=f'<b>Clean Environment (σ={clean_std})</b>',
            x=strat_labels,
            y=clean_succ_rates,
            marker=dict(color='#2B5C8F', line=dict(color='#1B3A5B', width=1.5))
        ))
        fig4.add_trace(go.Bar(
            name=f'<b>Noisy Environment (σ={noisy_std})</b>',
            x=strat_labels,
            y=noisy_succ_rates,
            marker=dict(
                color='#D95F02',
                pattern=dict(shape='/', fillmode='replace', fgcolor='#FFFFFF', fgopacity=0.35, size=8),
                line=dict(color='#8C3800', width=1.5)
            )
        ))

        fig4.update_layout(
            template='plotly_white',
            title=dict(
                text=f'<b>Figure 4: Prompt Scaffolding Ablation on {model_name} — {dim}D</b><br><sup>Empirical Target Success Rate Across Scaffolding Paradigms in Clean vs. Noisy Regimes</sup>',
                x=0.02, y=0.96,
                font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
            ),
            xaxis=dict(title='<b>Prompt Scaffolding Strategy</b>', tickfont=dict(size=12)),
            yaxis=dict(
                title='<b>Target Success Rate (Δy ≤ 10⁻⁸)</b>', range=[0, 1.10],
                showgrid=True, gridwidth=1, gridcolor='#EAEAEA'
            ),
            barmode='group', bargap=0.25, bargroupgap=0.1,
            width=850, height=520,
            margin=dict(l=65, r=30, t=95, b=65),
            legend=dict(
                orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
                bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
            )
        )

        out_p = model_fig_dir(model_name, dim) / 'figure_4_scaffolding.png'
        fig4.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 4 generated for all multi-strategy models.')


2026-08-24 23:32:27 INFO Chromium init'ed with kwargs {}
2026-08-24 23:32:27 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 23:32:27 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5u_mu79p.
2026-08-24 23:32:27 INFO Opening browser.
2026-08-24 23:32:27 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppibcch2z.
2026-08-24 23:32:27 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppibcch2z
2026-08-24 23:32:28 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5u_mu79p/index.html
2026-08-24 23:32:29 INFO Getting tab from queue (has 1)
2026-08-24 23:32:29 INFO Got A39C
2026-08-24 23:32:29 INFO Reloading tab A39C before return.
2026-08-24 23:32:29 INFO Putting tab A39C back (queue size: 0).
2026-08-24 23:32:29 INFO Waiting for all cleanups to finish.
2026-08-24 23:32:29 INFO Exiting Kaleido.
2026-08-24 23:32:29 INFO T

✅ Thesis Figure 4 generated for all multi-strategy models.
